# GenLab — Launcher
Plataforma modular para ejecutar modelos open source de IA en Colab.

**Antes de ejecutar:**
1. Concede permisos de Drive cuando se solicite.
2. Configura `HF_TOKEN` en Secrets de Colab (Icono llave → `HF_TOKEN`).
3. Asegúrate de usar runtime **T4 GPU** (Entorno de ejecución → Cambiar tipo de entorno de ejecución).

In [ ]:
# 1. Instalar dependencias
!pip install -q torch diffusers transformers huggingface-hub imageio imageio-ffmpeg accelerate psutil pyyaml

In [ ]:
# 2. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Clonar / actualizar repositorio
# CAMBIA esta URL por la de tu repositorio en GitHub
REPO_URL = 'https://github.com/tu-usuario/genlab.git'

import os
GENLAB_DIR = '/content/drive/MyDrive/GenLab'
os.makedirs(GENLAB_DIR, exist_ok=True)

if os.path.isdir(f'{GENLAB_DIR}/genlab'):
    %cd $GENLAB_DIR/genlab
    !git pull
else:
    %cd $GENLAB_DIR
    !git clone $REPO_URL genlab
    %cd genlab

!pip install -e .

In [ ]:
# 4. Bootstrap — diagnóstico del entorno
from genlab import GenLab
info = GenLab.bootstrap()

In [ ]:
# 5. Configurar token de Hugging Face
# Opción A: desde Secrets de Colab (recomendado)
from google.colab import userdata
import os
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN cargado desde Secrets de Colab.')
except userdata.SecretNotFoundError:
    # Opción B: preguntar al usuario
    token = input('Ingresa tu token de Hugging Face (o presiona Enter para omitir): ').strip()
    if token:
        os.environ['HF_TOKEN'] = token
        print('HF_TOKEN configurado manualmente.')
    else:
        print('HF_TOKEN no configurado. Si el modelo requiere autenticación, la descarga fallará.')

In [ ]:
# 6. Generar video
result = GenLab().run(
    task='text_to_video',
    model='cogvideo',
    prompt='Un astronauta montando un caballo en la luna, estilo cinematográfico',
    config={'model': {'steps': 50, 'fps': 8, 'frames': 49}}
)
print(f'Video generado: {result["video_path"]}')

In [ ]:
# 7. Mostrar resultado
from IPython.display import Video
Video(result['video_path'], width=480)